In [8]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-10-17 16:58:56--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-10-17 16:58:56 (41.5 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [9]:
with open("input.txt","r", encoding = "utf-8") as f:
  text = f.read()

In [3]:
print(len(text))

1115394


In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [16]:
chars = sorted(list(set("".join(text))))
stoi = {s:i for i,s in enumerate(chars)}
itos = {i:s for i,s in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: [itos[i] for i in l]

In [19]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.type, data.shape)
print(data[:1000])

<built-in method type of Tensor object at 0x7c4e093cc5a0> torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1234)

class BigramLanguageModel(nn.Module):


    def __init__(self, vocab_size):

          super().__init__()
          self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

    def forward(self,idx, targets = None):

        logits = self.token_embedding_table(idx) #(B,T,C)

        if targets is None:
          loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)

        return logits, loss


    def generate(self,idx,max_new_tokens):


        for _ in range(max_new_tokens):
           logits, loss = self(idx)
           logits = logits[:, -1, :]
           probs = F.softmax(logits, dim = -1)
           idx_next = torch.multinomial(probs,num_samples=1)
           idx = torch.cat((idx, idx_next), dim = 1)

        return idx


# ***1. GÖREV***

**Metni byte'lara çevir. Türkçe harfler içeren bir cümle seç (ç, ğ, ı, ö, ş, ü). Her harfin Unicode kod noktasını ve UTF-8 byte'larını yazdır. Hangi harfler 1 byte, hangileri 2 byte tutuyor? Kod noktalarını doğrudan vocab olarak kullanmak neden kötü bir fikir, kendi cümlelerinle yaz. (0:14:56 - 0:23:50)**


In [1]:
ord("ş") # karakterin unicode numarası evrensel

351

Bu işlem “Ş harfinin Unicode kataloğundaki numarası ne?” diye soruyor

Peki UTF-8 ne yapıyor? Önce küçük bir saklama kutusu düşünelim.
Bilgisayarda veriyi saklamak için kullandığımız birimlerden biri byte.
Şimdilik byte’ı şöyle düşün:
İçine yalnızca 0 ile 255 arasında bir tam sayı koyabildiğimiz küçük bir kutu.

Örneğin bir kutuya 97 koyabiliriz. Ama 351 koyamayız; kutunun sınırını aşıyor.
O zaman karakterleri bu kutulara yerleştirmek için ortak bir kurala ihtiyacımız var.
UTF-8, karakterleri byte kutularına nasıl yerleştireceğimizi belirleyen kuraldır.

In [4]:
list("ç".encode("utf-8")) # 2byte

[195, 167]

list("ş".encode("utf-8"))
# [197, 159]

0-255

| Saydığımız şey | Sayısı |
|---|---:|
| Karakter | 2 |
| Byte | 3 |
| Token — her byte’ı ayrı token seçersek | 3 |

Token, modeli beslemek için seçtiğimiz birim

Önceki GPT’nde birimi karakter seçmiştik. Şimdi byte seçmeyi düşünüyoruz.

Byte’ı birim seçersek başlangıç sözlüğümüz yalnızca 256 farklı token içerir: 0–255. Fakat bir karakter birkaç byte tuttuğunda modelin işleyeceği dizi uzar.

| Birleştirmeden önce | Birleştirmeden sonra |
|---|---|
| `[197, 159, 197, 159]` | `[256, 256]` |
| 4 token | 2 token |

Burada 256 bir token ID’si, byte değeri değil. Bu yüzden 255’ten büyük olmasında sorun yok.

In [3]:
list("a".encode("utf-8")) #1 byte

[97]

ş [197, 159] [197, 159] ş

list("ş".encode("utf-8")) bytes([197, 159]).decode("utf-8") encode bytlear decode harflere dönüşütrür

| Karakter | UTF-8 byte’ları | Kaç byte? |
|---|---|---:|
| `a` | `[97]` | 1 |
| `ş` | `[197, 159]` | 2 |

| Yöntem | Başlangıç sözlüğüne ne koyuyoruz? |
|---|---|
| Her karakter ayrı token | Desteklemek istediğimiz her harfi, sembolü, emojiyi |
| Her byte ayrı token | Yalnızca `0–255` arasındaki 256 değeri |

In [5]:
text = "Çağan şu gölün sıcaklığına bak."

In [6]:
print(text)

Çağan şu gölün sıcaklığına bak.


In [5]:
for ch in text:
  code_point = ord(ch)
  utf8_bytes =list(ch.encode("utf-8"))
  byte_count = len(utf8_bytes)


  print(
        repr(ch),
        "| Unicode:", code_point,
        "| UTF-8:", utf8_bytes,
        "| Byte sayısı:", byte_count
    )

'Ç' | Unicode: 199 | UTF-8: [195, 135] | Byte sayısı: 2
'a' | Unicode: 97 | UTF-8: [97] | Byte sayısı: 1
'ğ' | Unicode: 287 | UTF-8: [196, 159] | Byte sayısı: 2
'a' | Unicode: 97 | UTF-8: [97] | Byte sayısı: 1
'n' | Unicode: 110 | UTF-8: [110] | Byte sayısı: 1
' ' | Unicode: 32 | UTF-8: [32] | Byte sayısı: 1
'ş' | Unicode: 351 | UTF-8: [197, 159] | Byte sayısı: 2
'u' | Unicode: 117 | UTF-8: [117] | Byte sayısı: 1
' ' | Unicode: 32 | UTF-8: [32] | Byte sayısı: 1
'g' | Unicode: 103 | UTF-8: [103] | Byte sayısı: 1
'ö' | Unicode: 246 | UTF-8: [195, 182] | Byte sayısı: 2
'l' | Unicode: 108 | UTF-8: [108] | Byte sayısı: 1
'ü' | Unicode: 252 | UTF-8: [195, 188] | Byte sayısı: 2
'n' | Unicode: 110 | UTF-8: [110] | Byte sayısı: 1
' ' | Unicode: 32 | UTF-8: [32] | Byte sayısı: 1
's' | Unicode: 115 | UTF-8: [115] | Byte sayısı: 1
'ı' | Unicode: 305 | UTF-8: [196, 177] | Byte sayısı: 2
'c' | Unicode: 99 | UTF-8: [99] | Byte sayısı: 1
'a' | Unicode: 97 | UTF-8: [97] | Byte sayısı: 1
'k' | Unicode: 

In [7]:
print("Karakter sayısı:", len(text))
print("UTF-8 byte sayısı:", len(text.encode("utf-8"))) #22 adet 1 byte 9 tane 2 byte lık olmuş

Karakter sayısı: 31
UTF-8 byte sayısı: 40


In [10]:


import requests
import time
import re
import unicodedata
import hashlib
import json
from bs4 import BeautifulSoup
from urllib.parse import unquote, quote

API = "https://tr.wikisource.org/w/api.php"

session = requests.Session()
session.headers.update({
    "User-Agent": "YZ50StudentCorpus/1.0 (educational text collection)"
})

def sayfayi_al(baslik):
    response = session.get(
        API,
        params={
            "action": "parse",
            "page": baslik,
            "prop": "text",
            "format": "json",
            "formatversion": 2,
            "redirects": 1,
        },
        timeout=60,
    )
    response.raise_for_status()
    result = response.json()

    if "error" in result:
        raise RuntimeError(result["error"])

    return BeautifulSoup(result["parse"]["text"], "html.parser")


yazar = sayfayi_al("Kişi:Ömer Seyfettin")

basliklar = []
hikaye_bolumunde = False

for element in yazar.find_all(["h2", "a"]):
    if element.name == "h2":
        heading = element.get_text(" ", strip=True)

        if "Hikâyeleri" in heading:
            hikaye_bolumunde = True
        elif hikaye_bolumunde:
            break

    elif hikaye_bolumunde:
        href = element.get("href", "")

        if not href.startswith("/wiki/"):
            continue
        if "new" in element.get("class", []):
            continue

        baslik = unquote(href.split("/wiki/", 1)[1])
        baslik = baslik.split("#")[0].replace("_", " ")


        if ":" not in baslik and baslik not in basliklar:
            basliklar.append(baslik)

if not basliklar:
    raise RuntimeError("Hikâye bağlantıları bulunamadı.")

print("Bulunan hikâye bağlantısı:", len(basliklar))


def metni_temizle(soup):
    for element in soup.select(
        "table, .navbox, .metadata, .ambox, "
        ".ws-license, .licenseContainer, "
        ".mw-editsection, .reference, .references, #toc"
    ):
        element.decompose()

    for br in soup.find_all("br"):
        br.replace_with("\n")

    paragraflar = []

    for p in soup.find_all("p"):
        text = p.get_text()
        text = unicodedata.normalize("NFC", text)
        text = text.replace("\xa0", " ")
        text = re.sub(r"[ \t]+", " ", text).strip()

        if text:
            paragraflar.append(text)

    return "\n\n".join(paragraflar)

hedef_bayt = 1_000_000
toplam_bayt = 0

metinler = []
kaynaklar = []
gorulen_metinler = set()

for baslik in basliklar:
    try:
        soup = sayfayi_al(baslik)
        metin = metni_temizle(soup)


        if len(metin) < 500:
            continue


        ozet = hashlib.sha256(metin.encode("utf-8")).hexdigest()
        if ozet in gorulen_metinler:
            continue

        gorulen_metinler.add(ozet)

        parca = baslik + "\n\n" + metin
        metinler.append(parca)
        toplam_bayt += len((parca + "\n\n").encode("utf-8"))

        kaynaklar.append({
            "eser": baslik,
            "yazar": "Ömer Seyfettin",
            "url": (
                "https://tr.wikisource.org/wiki/"
                + quote(baslik.replace(" ", "_"), safe="")
            ),
        })

        print(f"{len(metinler):>3}. {baslik} — {toplam_bayt / 1e6:.2f} MB")

        if toplam_bayt >= hedef_bayt:
            break

    except (requests.RequestException, RuntimeError, ValueError) as error:
        print(f"Alınamadı: {baslik} — {error}")

    finally:
        time.sleep(0.5)



if not metinler:
    raise RuntimeError("Metin indirilemedi; eğitim aşamasına geçme.")

turkce_metin = "\n\n".join(metinler)

with open("input_tr.txt", "w", encoding="utf-8") as f:
    f.write(turkce_metin)

with open("turkce_kaynaklar.json", "w", encoding="utf-8") as f:
    json.dump(kaynaklar, f, ensure_ascii=False, indent=2)

print("\n--- VERİ ÖZETİ ---")
print("Hikâye sayısı:", len(metinler))
print("Dosya boyutu:", round(len(turkce_metin.encode("utf-8")) / 1e6, 3), "MB")
print("Karakter sayısı:", len(turkce_metin))
print("Farklı karakter sayısı:", len(set(turkce_metin)))

print("\n--- İLK 1000 KARAKTER ---")
print(turkce_metin[:1000])

if toplam_bayt < hedef_bayt:
    print("\nHedef boyuta ulaşılamadı; ek eser seçmemiz gerekiyor.")

Bulunan hikâye bağlantısı: 64
  1. Antiseptik — 0.00 MB
  2. Aşk Dalgası — 0.02 MB
  3. Aşk ve Ayak Parmakları — 0.03 MB
  4. Balkon (hikâye) — 0.04 MB
  5. Başını Vermeyen Şehit — 0.05 MB
  6. Binecek Şey — 0.07 MB
  7. Bir Hatıra — 0.07 MB
  8. Bir Kayışın Tesiri — 0.08 MB
  9. Bir Refikin Defter-i İhtisasatından — 0.08 MB
Alınamadı: Bomba — 429 Client Error: Too Many Requests for url: https://tr.wikisource.org/w/api.php?action=parse&page=Bomba&prop=text&format=json&formatversion=2&redirects=1
Alınamadı: Buse-i Mader — 429 Client Error: Too Many Requests for url: https://tr.wikisource.org/w/api.php?action=parse&page=Buse-i+Mader&prop=text&format=json&formatversion=2&redirects=1
Alınamadı: Büyücü — 429 Client Error: Too Many Requests for url: https://tr.wikisource.org/w/api.php?action=parse&page=B%C3%BCy%C3%BCc%C3%BC&prop=text&format=json&formatversion=2&redirects=1
Alınamadı: Cimnastiğe Dair - 2 — 429 Client Error: Too Many Requests for url: https://tr.wikisource.org/w/api.php?action

In [12]:
from pathlib import Path

cache = Path("turkce_metin_cache")
cache.mkdir(exist_ok=True)

def metni_temizle_duzeltilmis(soup):
    for element in soup.select(
        "table, .navbox, .metadata, .ambox, "
        ".ws-license, .licenseContainer, "
        ".mw-editsection, .reference, .references, #toc"
    ):
        element.decompose()

    for br in soup.find_all("br"):
        br.replace_with("\n")

    parcalar = []

    etiketler = ["p", "dd", "dt", "li"]

    for element in soup.find_all(etiketler):

        if element.find_parent(etiketler) is not None:
            continue

        metin = element.get_text()
        metin = unicodedata.normalize("NFC", metin)
        metin = metin.replace("\xa0", " ")
        metin = re.sub(r"[ \t]+", " ", metin).strip()

        if metin:
            parcalar.append(metin)

    return "\n\n".join(parcalar)


metinler = []
kaynaklar = []
gorulen_metinler = set()
toplam_bayt = 0

for baslik in basliklar:
    dosya_adi = hashlib.sha256(
        baslik.encode("utf-8")
    ).hexdigest() + ".txt"

    cache_dosyasi = cache / dosya_adi

    if cache_dosyasi.exists():
        metin = cache_dosyasi.read_text(encoding="utf-8")

    else:

        time.sleep(10)

        try:
            soup = sayfayi_al(baslik)
            metin = metni_temizle_duzeltilmis(soup)
            cache_dosyasi.write_text(metin, encoding="utf-8")

        except requests.HTTPError as error:
            if error.response.status_code == 429:
                print("\nSite yeniden hız sınırı uyguladı; durduruldu.")
                print("İnen metinler önbellekte duruyor.")
                print(
                    "Sunucunun Retry-After bilgisi:",
                    error.response.headers.get("Retry-After", "Belirtilmemiş")
                )
                break

            print("Alınamadı:", baslik)
            continue

        except (requests.RequestException, RuntimeError, ValueError) as error:
            print("Alınamadı:", baslik, str(error)[:120])
            continue

    if len(metin) < 500:
        continue

    ozet = hashlib.sha256(metin.encode("utf-8")).hexdigest()
    if ozet in gorulen_metinler:
        continue

    gorulen_metinler.add(ozet)
    parca = baslik + "\n\n" + metin
    metinler.append(parca)

    toplam_bayt += len((parca + "\n\n").encode("utf-8"))

    kaynaklar.append({
        "eser": baslik,
        "yazar": "Ömer Seyfettin",
        "url": "https://tr.wikisource.org/wiki/"
               + quote(baslik.replace(" ", "_"), safe=""),
    })

    print(f"{len(metinler)}. {baslik}: {toplam_bayt / 1e6:.3f} MB")

    if toplam_bayt >= 1_000_000:
        break


if metinler:
    turkce_metin = "\n\n".join(metinler)

    Path("input_tr.txt").write_text(turkce_metin, encoding="utf-8")

    with open("turkce_kaynaklar.json", "w", encoding="utf-8") as f:
        json.dump(kaynaklar, f, ensure_ascii=False, indent=2)

    print("\n--- VERİ ÖZETİ ---")
    print("Hikâye sayısı:", len(metinler))
    print("Boyut:", round(len(turkce_metin.encode("utf-8")) / 1e6, 3), "MB")
    print("Karakter sayısı:", len(turkce_metin))
    print("Farklı karakter sayısı:", len(set(turkce_metin)))

    print("\n--- İLK 1000 KARAKTER ---")
    print(turkce_metin[:1000])

    if toplam_bayt < 1_000_000:
        print("\nHenüz hedef boyutta değil; eğitime geçme.")
else:
    print("Yeni metin alınamadı. Mevcut dosyan değiştirilmedi.")

1. Antiseptik: 0.005 MB
2. Aşk Dalgası: 0.027 MB
3. Aşk ve Ayak Parmakları: 0.035 MB
4. Balkon (hikâye): 0.050 MB
5. Başını Vermeyen Şehit: 0.070 MB
6. Binecek Şey: 0.081 MB
7. Bir Hatıra: 0.091 MB
8. Bir Kayışın Tesiri: 0.096 MB
9. Bir Refikin Defter-i İhtisasatından: 0.098 MB
10. Bomba: 0.134 MB
11. Buse-i Mader: 0.135 MB
12. Büyücü: 0.150 MB
13. Cimnastiğe Dair - 2: 0.165 MB
14. Çakmak: 0.173 MB
15. Çirkinliğin Esrarı: 0.186 MB
16. Diyet: 0.205 MB
17. Elma: 0.209 MB
18. Falaka: 0.222 MB
19. Ferman: 0.246 MB
20. Fon Sadriştayn’ın Karısı: 0.271 MB
21. Fon Sadriştayn’ın Oğlu: 0.297 MB
22. Forsa: 0.306 MB
23. Havyar: 0.313 MB
24. Hediye...: 0.317 MB
25. Hürriyet Bayrakları: 0.335 MB
26. İffet: 0.336 MB
27. İlk Cinayet: 0.342 MB
28. İlk Namaz: 0.351 MB
29. İki Mebus: 0.367 MB
30. İlkbahar: 0.370 MB
31. Kaşağı: 0.377 MB
32. Keramet: 0.381 MB
33. Kesik Bıyık: 0.386 MB
34. Kır Sineği: 0.388 MB
35. Kıskançlık: 0.401 MB
36. Korkunç Bir Ceza: 0.408 MB
37. Kurbağa Duası: 0.420 MB
38. Kurumuş Ağ

# ***GÖREV 2***

**BPE'yi sıfırdan yaz ve kendi Türkçe metninde eğit. Geçen hafta derlediğin Türkçe metni kullan. Ardışık ikilileri say, en sık geçeni yeni bir token'la birleştir, istediğin vocab boyutuna gelene kadar tekrarla. İlk 20 birleştirmeyi metin olarak yazdır: hangi harf grupları önce birleşiyor? Sıkıştırma oranını (byte sayısı / token sayısı) yaz. (0:23:50 - 0:39:20)**

In [4]:
with open("input_tr.txt", "r", encoding="utf-8") as f:
    text = f.read()

tokens = list(text.encode("utf-8"))

In [5]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids, pair, idx): # işlenecek token bilgisi,birleştireiecek çift ve bu çiftin yerine yazılacak toeken id si
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

In [6]:
vocab_size = 276
num_merges = vocab_size - 256
ids = list(tokens)

merges = {}
vocab = {idx: bytes([idx]) for idx in range(256)}

for i in range(num_merges):
    stats = get_stats(ids)
    if not stats:
        break

    pair = max(stats, key=stats.get)
    idx = 256 + i

    ids = merge(ids, pair, idx)
    merges[pair] = idx
    vocab[idx] = vocab[pair[0]] + vocab[pair[1]]

    if i < 20:
        a = vocab[pair[0]].decode("utf-8", errors="backslashreplace")
        b = vocab[pair[1]].decode("utf-8", errors="backslashreplace")
        merged = vocab[idx].decode("utf-8", errors="backslashreplace")

        print(f"{i+1}. {a!r} + {b!r} -> {merged!r} | token {idx}")

1. '\\xc4' + '\\xb1' -> 'ı' | token 256
2. 'n' + ' ' -> 'n ' | token 257
3. '\\xc3' + '\\xbc' -> 'ü' | token 258
4. 'a' + 'r' -> 'ar' | token 259
5. '.' + ' ' -> '. ' | token 260
6. '\\xc5' + '\\x9f' -> 'ş' | token 261
7. ',' + ' ' -> ', ' | token 262
8. 'e' + ' ' -> 'e ' | token 263
9. 'e' + 'r' -> 'er' | token 264
10. 'a' + ' ' -> 'a ' | token 265
11. 'i' + ' ' -> 'i ' | token 266
12. 'i' + 'r' -> 'ir' | token 267
13. '\\xc3' + '\\xa7' -> 'ç' | token 268
14. 'i' + 'n' -> 'in' | token 269
15. '\\xc4' + '\\x9f' -> 'ğ' | token 270
16. 'a' + 'n' -> 'an' | token 271
17. '\n' + '\n' -> '\n\n' | token 272
18. 'a' + 'l' -> 'al' | token 273
19. 'a' + 'k' -> 'ak' | token 274
20. 'ı' + ' ' -> 'ı ' | token 275


In [7]:
print("tokens length:", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

tokens length: 748393
ids length: 584353
compression ratio: 1.28X


# ***GÖREV 3***

**encode ve decode yaz. decode(encode(metin)) metnin kendisini vermeli, Türkçe karakterli ve emoji içeren bir örnekle göster. Bazı token dizileri geçerli UTF-8 değil: decode bu durumda ne yapmalı, neden? (0:42:47 - 0:57:36)**

In [8]:
vocab = {idx: bytes([idx]) for idx in range(256)}

for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

In [9]:
def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text

print(decode([128]))

�


In [10]:
def encode(text):
    tokens = list(text.encode("utf-8"))

    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))

        if pair not in merges:
            break

        idx = merges[pair]
        tokens = merge(tokens, pair, idx)

    return tokens

print(encode(""))

[]


In [12]:
metin = "Çağan şu gölün sıcaklığına bak.! 😊🌿"

encoded = encode(metin)
decoded = decode(encoded)

print("Token'lar:", encoded)
print("Çözülen metin:", decoded)
print("Aynı mı?", decoded == metin)

Token'lar: [195, 135, 97, 270, 97, 257, 261, 117, 32, 103, 195, 182, 108, 258, 257, 115, 256, 99, 274, 108, 256, 270, 256, 110, 265, 98, 274, 46, 33, 32, 240, 159, 152, 138, 240, 159, 140, 191]
Çözülen metin: Çağan şu gölün sıcaklığına bak.! 😊🌿
Aynı mı? True


# ***GÖREV 4***

**GPT-4'ün regex desenini ekle. Metni önce GPT-4'ün deseniyle parçalara böl, BPE'yi her parçada ayrı çalıştır. Tokenizer'ı yeniden eğit, birleştirmeleri öncekiyle karşılaştır. Sonra "Ankara'dan" ve "İstanbul'da" gibi kesme işaretli kelimeleri bu desenle böl. Desen İngilizce için yazılmış: Türkçede ne yapıyor, kendi cümlelerinle yaz. (0:57:36 - 1:11:38)**


In [15]:
import regex as re

pat = re.compile(
    r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}++|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s"""
)


parts = pat.findall(text)
chunks = [list(part.encode("utf-8")) for part in parts]

print(parts[:20])

['Antiseptik', '\n\n', 'Minimini', ',', ' güzel', ',', ' şeytan', ' Bedia', "'yı", ' ailesi', ' büyük', ' bir', ' adama', ' vermek', ' istiyordu', '.', ' Halbuki', ' o', ' iki', ' senedir']


In [16]:
vocab_size = 276
regex_merges = {}
regex_vocab = {idx: bytes([idx]) for idx in range(256)}

for i in range(vocab_size - 256):
    stats = {}

    for chunk in chunks:
        for pair, count in get_stats(chunk).items():
            stats[pair] = stats.get(pair, 0) + count

    if not stats:
        break

    pair = max(stats, key=stats.get)
    idx = 256 + i

    chunks = [merge(chunk, pair, idx) for chunk in chunks]

    regex_merges[pair] = idx
    regex_vocab[idx] = regex_vocab[pair[0]] + regex_vocab[pair[1]]

In [17]:
old_rules = list(merges.items())[:20]
new_rules = list(regex_merges.items())[:20]

for i, (old_rule, new_rule) in enumerate(zip(old_rules, new_rules), 1):
    old_pair, old_idx = old_rule
    new_pair, new_idx = new_rule

    old_text = vocab[old_idx].decode("utf-8", errors="backslashreplace")
    new_text = regex_vocab[new_idx].decode("utf-8", errors="backslashreplace")

    print(f"{i}. Önce: {old_text!r} | Regex ile: {new_text!r}")

regex_token_count = sum(len(chunk) for chunk in chunks)

print(f"Önceki oran: {len(tokens) / len(ids):.2f}X")
print(f"Regex ile oran: {len(tokens) / regex_token_count:.2f}X")

1. Önce: 'ı' | Regex ile: 'ı'
2. Önce: 'n ' | Regex ile: 'ü'
3. Önce: 'ü' | Regex ile: 'ar'
4. Önce: 'ar' | Regex ile: ' b'
5. Önce: '. ' | Regex ile: 'ş'
6. Önce: 'ş' | Regex ile: 'er'
7. Önce: ', ' | Regex ile: 'in'
8. Önce: 'e ' | Regex ile: 'an'
9. Önce: 'er' | Regex ile: ' k'
10. Önce: 'a ' | Regex ile: 'en'
11. Önce: 'i ' | Regex ile: 'ın'
12. Önce: 'ir' | Regex ile: 'ir'
13. Önce: 'ç' | Regex ile: 'ç'
14. Önce: 'in' | Regex ile: ' d'
15. Önce: 'ğ' | Regex ile: ' g'
16. Önce: 'an' | Regex ile: 'ğ'
17. Önce: '\n\n' | Regex ile: '\n\n'
18. Önce: 'al' | Regex ile: 'al'
19. Önce: 'ak' | Regex ile: ' s'
20. Önce: 'ı ' | Regex ile: 'ak'
Önceki oran: 1.28X
Regex ile oran: 1.26X


In [18]:
words = ["Ankara'dan", "İstanbul'da", "Ankara’dan", "İstanbul’da"]

for word in words:
    print(word, "->", pat.findall(word))

Ankara'dan -> ['Ankara', "'d", 'an']
İstanbul'da -> ['İstanbul', "'d", 'a']
Ankara’dan -> ['Ankara', '’dan']
İstanbul’da -> ['İstanbul', '’da']


# ***GÖREV 5***

**Türkçe ile İngilizceyi karşılaştır. Aynı paragrafın Türkçesini ve İngilizcesini al. GPT-4'ün tokenizer'ıyla (tiktoken, "cl100k_base") ikisini de token'la, kelime başına kaç token düştüğünü yaz. Sonra aynı Türkçe paragrafı kendi tokenizer'ınla token'la. Türkçenin eklemeli yapısı bu sayıları nasıl etkiliyor?**


In [19]:
%pip install tiktoken

In [20]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

tr_text = (
    "Bugün arkadaşlarımla kütüphaneye gittim. "
    "Derslerimizde öğrendiğimiz konuları birlikte tekrar ettik. "
    "Anlamadığımız soruları öğretmenimize sormak için defterlerimize yazdık. "
    "Çalışmamız bittikten sonra eve döndük."
)

en_text = (
    "Today I went to the library with my friends. "
    "We reviewed the topics we had learned in our classes together. "
    "We wrote the questions we did not understand in our notebooks "
    "to ask our teacher. After finishing our work, we returned home."
)

In [21]:
tr_tokens = enc.encode(tr_text)
en_tokens = enc.encode(en_text)

tr_words = len(tr_text.split())
en_words = len(en_text.split())

print("Türkçe kelime sayısı:", tr_words)
print("Türkçe token sayısı:", len(tr_tokens))
print(f"Türkçe token/kelime: {len(tr_tokens) / tr_words:.2f}")

print()

print("İngilizce kelime sayısı:", en_words)
print("İngilizce token sayısı:", len(en_tokens))
print(f"İngilizce token/kelime: {len(en_tokens) / en_words:.2f}")

Türkçe kelime sayısı: 22
Türkçe token sayısı: 84
Türkçe token/kelime: 3.82

İngilizce kelime sayısı: 42
İngilizce token sayısı: 47
İngilizce token/kelime: 1.12


In [22]:
def encode_regex(text):
    ids = []

    for part in pat.findall(text):
        tokens = list(part.encode("utf-8"))

        while len(tokens) >= 2:
            stats = get_stats(tokens)
            pair = min(
                stats,
                key=lambda p: regex_merges.get(p, float("inf"))
            )

            if pair not in regex_merges:
                break

            idx = regex_merges[pair]
            tokens = merge(tokens, pair, idx)

        ids.extend(tokens)

    return ids


my_tokens = encode_regex(tr_text)

print("Kendi tokenizer'ımın token sayısı:", len(my_tokens))
print(f"Kendi tokenizer'ım token/kelime: {len(my_tokens) / tr_words:.2f}")

Kendi tokenizer'ımın token sayısı: 187
Kendi tokenizer'ım token/kelime: 8.50
